In [2]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# 1. Setup
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

# 2. Load Reviews (Sample 50k to keep it fast)
print("Processing reviews... this takes about 1 minute.")
reviews = pd.read_csv('../data/raw/reviews.csv.gz', nrows=50000)
reviews = reviews.dropna(subset=['comments'])

# 3. Calculate Sentiment
def get_sentiment(text):
    return sia.polarity_scores(str(text))['compound']

reviews['sentiment_score'] = reviews['comments'].apply(get_sentiment)

# 4. Group by listing_id
df_sentiment = reviews.groupby('listing_id')['sentiment_score'].mean().reset_index()

print("Sentiment Scores Created!")
df_sentiment.head()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jnanadeep/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Processing reviews... this takes about 1 minute.
Sentiment Scores Created!


,listing_id,sentiment_score
0,17878,0.593567
1,25026,0.556820
2,35764,0.340955
3,48305,0.553079
4,48901,0.009629


In [6]:
# 1. Load Listings
# (Make sure listings.csv.gz is also uploaded or downloaded)
df_listings = pd.read_csv('../data/raw/listings.csv')

# 2. Clean Price and Features
df_listings['price'] = df_listings['price'].str.replace('$', '').str.replace(',', '').astype(float)
df_listings = df_listings[(df_listings['price'] >= 50) & (df_listings['price'] <= 5000)]
df_listings['bedrooms'] = df_listings['bedrooms'].fillna(1)
df_listings['accommodates'] = df_listings['accommodates'].fillna(1)

# 3. MERGE: Add Sentiment to Listings
# We match 'id' from listings with 'listing_id' from reviews
df_final = pd.merge(df_listings, df_sentiment, left_on='id', right_on='listing_id', how='left')

# If a house has no reviews, give it 0 (neutral)
df_final['sentiment_score'] = df_final['sentiment_score'].fillna(0)

print("Merging Complete! Your data is now Multimodal.")

Merging Complete! Your data is now Multimodal.


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Pick our Features (Numbers only)
features_list = ['accommodates', 'bedrooms', 'number_of_reviews', 'sentiment_score']
X = df_final[features_list]
y = df_final['price']

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Model
print("Training the Final Model...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluate
preds = model.predict(X_test)
print(f"\n--- RESULTS FOR PPT ---")
print(f"Mean Absolute Error: R$ {mean_absolute_error(y_test, preds):.2f}")
print(f"R2 Score (Accuracy): {r2_score(y_test, preds):.2f}")

Training the Final Model...

--- RESULTS FOR PPT ---
Mean Absolute Error: R$ 328.87
R2 Score (Accuracy): 0.26
